In [1]:
from __future__ import annotations

import sys
from pathlib import Path
import time

project_dir = Path.cwd()

# Modified .py files can be placed either in the same folder as this notebook
# or inside a subfolder named FHF_modified.
module_dir = project_dir
if not (module_dir / "FHF_runner.py").exists():
    candidate = project_dir / "FHF_modified"
    if (candidate / "FHF_runner.py").exists():
        module_dir = candidate

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from FHF_runner import Args, set_seed, run_fhf
from FHF_nf import load_clients_from_csv
from FHF_Client import Client

import warnings
warnings.filterwarnings("ignore", message=".*flash attention.*")

In [2]:
print("torch version:", torch.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("working directory:", project_dir)
print("module directory:", module_dir)

torch version: 2.3.1+cu118
device: cuda
working directory: E:\yjz\FHF\FHF_gef2017
module directory: E:\yjz\FHF\FHF_gef2017


In [3]:
# -------------------------
# Parameter setup
# -------------------------
args = Args()
args.seed = 42
set_seed(args.seed, torch_num_threads=getattr(args, "torch_num_threads", 1))
args.device = "cuda" if torch.cuda.is_available() else "cpu"

args.csv_path = "E:/yjz/Datasets/GEFCOM2017/Gef2017_univarivate.csv" 
#"E:/yjz/Datasets/AusGrid2013/AusGrid2013_univariate.csv"
args.model_type = "lstm"
args.truncated = None
args.fh = 0
args.lags = 48
args.num_rounds = 100 + args.fh * 50

args.recon_method = [
    "bu",
    "mint_cov",
    "mint_shrinkage",
    "mint_ols",
    "mint_var",
    "wls_structure",
]

# -------------------------
# Early stopping settings
# -------------------------
args.early_stop_enabled = True
args.early_stop_tol = 1e-5
args.early_stop_metric = "avg_normalised_loss_delta"
args.min_rounds = 20 + args.fh * 50

# Logging / output format
args.verbose_rounds = True
args.print_every = 1
args.return_result = True

args


Args(csv_path='E:/yjz/Datasets/GEFCOM2017/Gef2017_univarivate.csv', partition_col='partition_id', series_col='unique_id', time_col='ds', target_col='y', truncated=None, lags=48, fh=0, test_ratio=0.2, batch_size=256, model_type='lstm', input_size=1, hidden_size=64, num_layers=2, dropout=0.1, c_in=1, d_model=128, n_heads=4, e_layers=3, device='cuda', num_rounds=100, local_epochs=1, lr=0.001, weight_decay=0.0, optimizer='adam', recon_method=['bu', 'mint_cov', 'mint_shrinkage', 'mint_ols', 'mint_var', 'wls_structure'], ridge=1e-06, td_eps=1e-08, seed=42, validate_p2p=True, p2p_atol=1e-08, torch_num_threads=1, early_stop_enabled=True, early_stop_tol=1e-05, early_stop_metric='avg_normalised_loss_delta', min_rounds=20, verbose_rounds=True, print_every=1, return_result=True)

In [4]:
# Optional quick data check before training
client_ids, cid2series_preview, cid2df_preview = load_clients_from_csv(
    csv_path=args.csv_path,
    partition_col=args.partition_col,
    series_col=args.series_col,
    time_col=args.time_col,
    target_col=args.target_col,
    lags=args.lags,
    fh=args.fh,
    truncated=args.truncated,
)

c = Client(client_ids[0], cid2series_preview[client_ids[0]], cid2df_preview[client_ids[0]], args)
c.prepare_data()

print("n_clients:", len(client_ids))
print("first client:", client_ids[0], cid2series_preview[client_ids[0]])
print("device:", c.device)
print("train samples:", c.num_train_samples(), "batches:", len(c.train_loader))

n_clients: 10
first client: 0 Total
device: cuda
train samples: 99288 batches: 388


In [5]:
# -------------------------
# Run FHF
# -------------------------
result = run_fhf(args)

clients = result["clients"]
cloud = result["cloud"]
cid2series = result["cid2series"]
timings = result["timings"]
round_logs = result["round_logs"]
early_stop = result["early_stop"]

print("early_stop:", early_stop)

Loaded 10 clients.
Recon methods: ['bu', 'mint_cov', 'mint_shrinkage', 'mint_ols', 'mint_var', 'wls_structure']
[Round 1/100] avg_normalised_loss=0.0113526, avg_actual_loss=12028.9, delta=nan, stopped=False
[Round 2/100] avg_normalised_loss=0.00701616, avg_actual_loss=7096.23, delta=0.00433645, stopped=False
[Round 3/100] avg_normalised_loss=0.00576262, avg_actual_loss=5037.78, delta=0.00125354, stopped=False
[Round 4/100] avg_normalised_loss=0.00480101, avg_actual_loss=4501.93, delta=0.000961608, stopped=False
[Round 5/100] avg_normalised_loss=0.00440283, avg_actual_loss=3637.6, delta=0.000398185, stopped=False
[Round 6/100] avg_normalised_loss=0.00386872, avg_actual_loss=3006.36, delta=0.000534101, stopped=False
[Round 7/100] avg_normalised_loss=0.00360791, avg_actual_loss=2854.48, delta=0.000260816, stopped=False
[Round 8/100] avg_normalised_loss=0.00339767, avg_actual_loss=2876.79, delta=0.000210241, stopped=False
[Round 9/100] avg_normalised_loss=0.00336531, avg_actual_loss=2636.8

In [6]:
# Cloud-round training log
round_log_df = pd.DataFrame(round_logs)
round_log_df.tail()

,round,num_rounds,avg_normalised_loss,avg_actual_loss,normalised_loss_delta,avg_train_loss,n_train_nodes,early_stop_enabled,early_stop_tol,early_stop_metric,stopped
15,16,100,0.002787,1927.069658,0.000161,0.002787,10,True,0.00001,avg_normalised_loss_delta,False
16,17,100,0.002733,1969.145311,0.000053,0.002733,10,True,0.00001,avg_normalised_loss_delta,False
17,18,100,0.002704,1978.417985,0.000029,0.002704,10,True,0.00001,avg_normalised_loss_delta,False
18,19,100,0.002720,1944.302307,0.000016,0.002720,10,True,0.00001,avg_normalised_loss_delta,False
19,20,100,0.002717,1781.427928,0.000003,0.002717,10,True,0.00001,avg_normalised_loss_delta,True


In [7]:
# Check reconciled forecast methods and shapes
cid0 = list(clients.keys())[0]
print("client:", cid0, cid2series[cid0])
print("methods:", list(clients[cid0].reconciled_forecasts.keys()))

for method, arr in clients[cid0].reconciled_forecasts.items():
    print(method, np.asarray(arr).shape)

client: 0 Total
methods: ['bu', 'mint_cov', 'mint_shrinkage', 'mint_ols', 'mint_var', 'wls_structure']
bu (24835, 1)
mint_cov (24835, 1)
mint_shrinkage (24835, 1)
mint_ols (24835, 1)
mint_var (24835, 1)
wls_structure (24835, 1)


## Level-wise evaluation: MAE, MASE, RMSSE

In [8]:
def _as_2d(arr):
    arr = np.asarray(arr, dtype=np.float64)
    if arr.ndim == 1:
        arr = arr[:, None]
    return arr


def _target_cols(args):
    return ['y'] + [f'post_{i}' for i in range(1, int(args.fh) + 1)]


def split_node_df(df_one, args):
    time_col = getattr(args, 'time_col', 'ds')
    sub = df_one.sort_values(time_col).copy().reset_index(drop=True)

    n_windows = len(sub)
    n_raw = n_windows + int(args.lags) + int(args.fh)
    n_train_raw = int((1 - float(args.test_ratio)) * n_raw)
    n_train_windows = n_train_raw - int(args.lags) - int(args.fh)

    if n_train_windows <= 0 or n_train_windows >= n_windows:
        raise ValueError(
            f'Bad split: n_windows={n_windows}, n_train_windows={n_train_windows}'
        )

    return sub.iloc[:n_train_windows].copy(), sub.iloc[n_train_windows:].copy()


def make_naive_forecast(node, args):
    """Simple random-walk naive forecast: use the latest available lag, lags_1.

    For multi-step horizons, the same last observed value is repeated across
    horizons. This is the standard persistence baseline and uses no test future
    information.
    """
    _, ts_df = split_node_df(node.df, args)
    y_true = _as_2d(node.test_true)
    if 'lags_1' not in ts_df.columns:
        raise KeyError("Cannot compute naive forecast because 'lags_1' is missing.")
    naive_1d = ts_df['lags_1'].to_numpy(dtype=np.float64)
    if len(naive_1d) != y_true.shape[0]:
        raise ValueError(
            f'Naive length mismatch: naive={len(naive_1d)}, y_true={y_true.shape[0]}'
        )
    return np.repeat(naive_1d[:, None], y_true.shape[1], axis=1)


def _safe_scale_denominators(train_true, eps=1e-12):
    """Return MASE and RMSSE denominators from training true values only."""
    train_true = _as_2d(train_true)
    if train_true.shape[0] <= 1:
        return np.nan, np.nan

    diff = np.diff(train_true, axis=0)
    mase_denom = np.nanmean(np.abs(diff))
    rmsse_denom = np.nanmean(diff ** 2)

    if (not np.isfinite(mase_denom)) or mase_denom <= eps:
        mase_denom = np.nan
    if (not np.isfinite(rmsse_denom)) or rmsse_denom <= eps:
        rmsse_denom = np.nan

    return float(mase_denom), float(rmsse_denom)


def _role_from_level(level, min_level, max_level):
    if level == min_level:
        return 'top'
    if level == max_level:
        return 'bottom'
    return 'middle'


def compute_metrics_by_level(clients, cid2series, args, include_naive=True, eps=1e-12):
    rows = []

    levels = {
        int(cid): int(str(series).count('/') + 1)
        for cid, series in cid2series.items()
    }
    min_level = min(levels.values())
    max_level = max(levels.values())

    for cid in sorted(clients.keys()):
        node = clients[cid]
        unique_id = str(cid2series[cid])
        level = int(unique_id.count('/') + 1)
        role = _role_from_level(level, min_level, max_level)

        y_true = _as_2d(node.test_true)
        train_true = _as_2d(node.y2)
        mase_denom, rmsse_denom = _safe_scale_denominators(train_true, eps=eps)

        method_forecasts = {'base': node.base_forecast}
        if include_naive:
            method_forecasts['naive'] = make_naive_forecast(node, args)
        method_forecasts.update(node.reconciled_forecasts)

        for method, forecast in method_forecasts.items():
            y_pred = _as_2d(forecast)
            if y_pred.shape != y_true.shape:
                raise ValueError(
                    f'Shape mismatch for cid={cid}, method={method}: '
                    f'y_true={y_true.shape}, y_pred={y_pred.shape}'
                )

            error = y_true - y_pred
            mae = float(np.nanmean(np.abs(error)))
            mse = float(np.nanmean(error ** 2))
            mase = mae / mase_denom if np.isfinite(mase_denom) else np.nan
            rmsse = np.sqrt(mse / rmsse_denom) if np.isfinite(rmsse_denom) else np.nan

            rows.append({
                'node_id': int(cid),
                'unique_id': unique_id,
                'level': level,
                'role': role,
                'method': str(method),
                'MAE': mae,
                'MASE': float(mase) if np.isfinite(mase) else np.nan,
                'RMSSE': float(rmsse) if np.isfinite(rmsse) else np.nan,
            })

    per_node_metrics = pd.DataFrame(rows)

    metrics_by_level = (
        per_node_metrics
        .groupby(['level', 'role', 'method'], as_index=False)
        .agg(
            MAE=('MAE', 'mean'),
            MASE=('MASE', 'mean'),
            RMSSE=('RMSSE', 'mean'),
            n_nodes=('node_id', 'nunique'),
        )
        .sort_values(['level', 'method'])
        .reset_index(drop=True)
    )

    overall_metrics = (
        per_node_metrics
        .groupby(['method'], as_index=False)
        .agg(
            MAE=('MAE', 'mean'),
            MASE=('MASE', 'mean'),
            RMSSE=('RMSSE', 'mean'),
            n_nodes=('node_id', 'nunique'),
        )
        .sort_values(['method'])
        .reset_index(drop=True)
    )

    return per_node_metrics, metrics_by_level, overall_metrics


def build_forecast_table_fhf(clients, cid2series, args, include_naive=True):
    """Build a per-series forecast table with base, reconciled, and naive forecasts."""
    rows = []
    target_cols = _target_cols(args)
    time_col = getattr(args, 'time_col', 'ds')

    for cid in sorted(clients.keys()):
        node = clients[cid]
        _, ts_df = split_node_df(node.df, args)
        unique_id = str(cid2series[cid])
        y_true = _as_2d(node.test_true)
        base = _as_2d(node.base_forecast)
        naive = make_naive_forecast(node, args) if include_naive else None
        rec = {method: _as_2d(arr) for method, arr in node.reconciled_forecasts.items()}

        for h_idx, target_col in enumerate(target_cols):
            for t_idx, (_, row) in enumerate(ts_df.iterrows()):
                out = {
                    'node_id': int(cid),
                    'unique_id': unique_id,
                    'ds': row[time_col],
                    'horizon_index': int(h_idx),
                    'target_col': target_col,
                    'y_true': float(y_true[t_idx, h_idx]),
                    'base': float(base[t_idx, h_idx]),
                }
                if include_naive:
                    out['naive'] = float(naive[t_idx, h_idx])
                for method, arr in rec.items():
                    out[str(method)] = float(arr[t_idx, h_idx])
                rows.append(out)

    return pd.DataFrame(rows)


def make_timing_df(timings):
    ordered_modules = [
        'setup_sec',
        'data_preparation_sec',
        'model_server_initialization_sec',
        'fl_training_sec',
        'residual_collection_sec',
        'recon_matrix_sec',
        'p2p_reconcile_sec',
        'reconciliation_sec',
        'evaluation_sec',
        'total_sec',
    ]
    return pd.DataFrame(
        [{'module': key, 'seconds': timings.get(key, np.nan)} for key in ordered_modules]
    )


def save_fhf_outputs(output_prefix, clients, cid2series, args, per_node_metrics,
                     metrics_by_level, overall_metrics, round_logs, timings):
    """Save forecasts, per-series metrics, aggregate metrics, round logs, and time costs."""
    forecast_table = build_forecast_table_fhf(clients, cid2series, args, include_naive=True)
    timing_df = make_timing_df(timings)

    paths = {
        'forecasts': f'{output_prefix}_forecasts.csv',
        'per_node_metrics': f'{output_prefix}_per_node_metrics.csv',
        'metrics_by_level': f'{output_prefix}_metrics_by_level.csv',
        'overall_metrics': f'{output_prefix}_overall_metrics.csv',
        'round_logs': f'{output_prefix}_round_logs.csv',
        'timing': f'{output_prefix}_timing.csv',
    }

    forecast_table.to_csv(paths['forecasts'], index=False)
    per_node_metrics.to_csv(paths['per_node_metrics'], index=False)
    metrics_by_level.to_csv(paths['metrics_by_level'], index=False)
    overall_metrics.to_csv(paths['overall_metrics'], index=False)
    pd.DataFrame(round_logs).to_csv(paths['round_logs'], index=False)
    timing_df.to_csv(paths['timing'], index=False)

    return paths, forecast_table, timing_df


_eval_t0 = time.perf_counter()
per_node_metrics, metrics_by_level, overall_metrics = compute_metrics_by_level(
    clients, cid2series, args, include_naive=True
)
timings['evaluation_sec'] = time.perf_counter() - _eval_t0
timings['total_sec'] = (
    timings.get('setup_sec', 0.0)
    + timings.get('fl_training_sec', 0.0)
    + timings.get('reconciliation_sec', 0.0)
    + timings.get('evaluation_sec', 0.0)
)

output_prefix = f'fh{args.fh + 1}_FHF'
output_paths, forecast_table, timing_df = save_fhf_outputs(
    output_prefix=output_prefix,
    clients=clients,
    cid2series=cid2series,
    args=args,
    per_node_metrics=per_node_metrics,
    metrics_by_level=metrics_by_level,
    overall_metrics=overall_metrics,
    round_logs=round_logs,
    timings=timings,
)

result['timings'] = timings
result['per_node_metrics'] = per_node_metrics
result['metrics_by_level'] = metrics_by_level
result['overall_metrics'] = overall_metrics
result['forecast_table'] = forecast_table
result['output_paths'] = output_paths

print('Saved outputs:')
for name, path in output_paths.items():
    print(f'  {name}: {path}')

metrics_by_level


Saved outputs:
  forecasts: fh1_FHF_forecasts.csv
  per_node_metrics: fh1_FHF_per_node_metrics.csv
  metrics_by_level: fh1_FHF_metrics_by_level.csv
  overall_metrics: fh1_FHF_overall_metrics.csv
  round_logs: fh1_FHF_round_logs.csv
  timing: fh1_FHF_timing.csv


,level,role,method,MAE,MASE,RMSSE,n_nodes
0,1,top,base,87.477074,0.151210,0.154257,1
1,1,top,bu,89.902809,0.155403,0.158801,1
2,1,top,mint_cov,87.568614,0.151369,0.154452,1
3,1,top,mint_ols,87.664589,0.151534,0.154608,1
4,1,top,mint_shrinkage,87.571780,0.151374,0.154456,1
5,1,top,mint_var,89.047656,0.153925,0.157206,1
6,1,top,naive,527.787525,0.912318,0.895358,1
7,1,top,wls_structure,87.664589,0.151534,0.154608,1
8,2,middle,base,18.636018,0.209778,0.212415,6
9,2,middle,bu,18.711702,0.210069,0.212710,6


In [9]:
# Optional overall average across all hierarchy nodes
overall_metrics

,method,MAE,MASE,RMSSE,n_nodes
0,base,25.392411,0.203420,0.207245,10
1,bu,25.680395,0.204014,0.207876,10
2,mint_cov,25.159788,0.201079,0.204503,10
3,mint_ols,25.297427,0.202071,0.205167,10
4,mint_shrinkage,25.160575,0.201079,0.204502,10
5,mint_var,25.489737,0.203026,0.206759,10
6,naive,130.514212,0.920111,0.898427,10
7,wls_structure,25.297425,0.202071,0.205167,10


## Timing report

In [10]:
# Timing report saved by save_fhf_outputs(...).
timing_df


,module,seconds
0,setup_sec,8.409178
1,data_preparation_sec,8.404470
2,model_server_initialization_sec,0.004704
3,fl_training_sec,1035.879574
4,residual_collection_sec,23.784835
5,recon_matrix_sec,0.223401
6,p2p_reconcile_sec,0.100904
7,reconciliation_sec,24.109140
8,evaluation_sec,1.083549
9,total_sec,1069.481441


In [11]:
# Compact run summary
summary = {
    'stopped_early': result['stopped_early'],
    'stop_reason': result['stop_reason'],
    'stopped_round': result['stopped_round'],
    'final_avg_normalised_loss': result['final_avg_normalised_loss'],
    'final_avg_actual_loss': result['final_avg_actual_loss'],
    'final_normalised_loss_delta': result['final_normalised_loss_delta'],
    'n_clients': len(clients),
    'methods': list(clients[cid0].reconciled_forecasts.keys()),
    'output_paths': output_paths,
}
summary


{'stopped_early': True,
 'stop_reason': 'abs(avg_normalised_loss - previous_avg_normalised_loss) < 1e-05',
 'stopped_round': 20,
 'final_avg_normalised_loss': 0.0027170383611742717,
 'final_avg_actual_loss': 1781.4279277150192,
 'final_normalised_loss_delta': 2.9494611230064527e-06,
 'n_clients': 10,
 'methods': ['bu',
  'mint_cov',
  'mint_shrinkage',
  'mint_ols',
  'mint_var',
  'wls_structure'],
 'output_paths': {'forecasts': 'fh1_FHF_forecasts.csv',
  'per_node_metrics': 'fh1_FHF_per_node_metrics.csv',
  'metrics_by_level': 'fh1_FHF_metrics_by_level.csv',
  'overall_metrics': 'fh1_FHF_overall_metrics.csv',
  'round_logs': 'fh1_FHF_round_logs.csv',
  'timing': 'fh1_FHF_timing.csv'}}